Koko aineisto ja sen käsittely dataframeiksi

Tähän lohkoon kerätään kaikki säähän liittyvät columnit ja muodostetaan niistä oma df
Extract weather columns into a dedicated DataFrame for further analysis

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Bigger").getOrCreate()

df = spark.read.csv("C:/Users/serko/US_Accidents/US_Accidents_March23.csv", header=True)

weather_cols = [
    "ID", "Weather_Timestamp", "Temperature(F)", "Wind_Chill(F)", "Humidity(%)",
    "Pressure(in)", "Visibility(mi)", "Wind_Direction", "Wind_Speed(mph)",
    "Precipitation(in)", "Weather_Condition", "Airport_Code"
]

weather_df = df.select(weather_cols)
weather_df.cache()
weather_df.count()





7728394

Lasketaan weather_df:n columnien nolla-arvot sekä prosenttiosuus 

In [3]:
from pyspark.sql.functions import col, sum, round, lit

# rivien kokonaismäärä
rows_total = weather_df.count()

# nullien määrä ja prosentit
null_counts = weather_df.select([
    sum(col(c).isNull().cast("int")).alias(f"{c}_nulls")
    for c in weather_df.columns
] + [
    round((sum(col(c).isNull().cast("int")) / rows_total) * 100, 2).alias(f"{c}_pct")
    for c in weather_df.columns
])

null_counts.show(truncate=False)


+--------+-----------------------+--------------------+-------------------+-----------------+------------------+--------------------+--------------------+---------------------+-----------------------+-----------------------+------------------+------+---------------------+------------------+-----------------+---------------+----------------+------------------+------------------+-------------------+---------------------+---------------------+----------------+
|ID_nulls|Weather_Timestamp_nulls|Temperature(F)_nulls|Wind_Chill(F)_nulls|Humidity(%)_nulls|Pressure(in)_nulls|Visibility(mi)_nulls|Wind_Direction_nulls|Wind_Speed(mph)_nulls|Precipitation(in)_nulls|Weather_Condition_nulls|Airport_Code_nulls|ID_pct|Weather_Timestamp_pct|Temperature(F)_pct|Wind_Chill(F)_pct|Humidity(%)_pct|Pressure(in)_pct|Visibility(mi)_pct|Wind_Direction_pct|Wind_Speed(mph)_pct|Precipitation(in)_pct|Weather_Condition_pct|Airport_Code_pct|
+--------+-----------------------+--------------------+-------------------+-

In [25]:
from pyspark.sql.functions import col
import builtins

rows_total = weather_df.count()

data = []
for c in weather_df.columns:
    n = weather_df.filter(col(c).isNull()).count()
    pct = builtins.round(n / rows_total * 100, 2)  # HUOM: builtins.round
    data.append((c, n, pct))

null_df = spark.createDataFrame(data, ["column", "nulls", "pct"])
null_df.show(truncate=False)


+-----------------+-------+-----+
|column           |nulls  |pct  |
+-----------------+-------+-----+
|ID               |0      |0.0  |
|Weather_Timestamp|120228 |1.56 |
|Temperature(F)   |163853 |2.12 |
|Wind_Chill(F)    |1999019|25.87|
|Humidity(%)      |174144 |2.25 |
|Pressure(in)     |140679 |1.82 |
|Visibility(mi)   |177098 |2.29 |
|Wind_Direction   |175206 |2.27 |
|Wind_Speed(mph)  |571233 |7.39 |
|Precipitation(in)|2203586|28.51|
|Weather_Condition|173459 |2.24 |
|Airport_Code     |22635  |0.29 |
+-----------------+-------+-----+



Säätilojen sanalliset kuvaukset

In [26]:
from pyspark.sql.functions import col, sum, when, lit


total_rows = df.count()

# Puuttuvien Weather_Condition-arvojen määrä
missing_weather = df.select(
    sum(when(col("Weather_Condition").isNull(), 1).otherwise(0)).alias("missing_weather")
)

# Lisätään total_rows samaan taulukkoon
result = missing_weather.withColumn("total_rows", lit(total_rows))

result.show(truncate=False)



+---------------+----------+
|missing_weather|total_rows|
+---------------+----------+
|173459         |7728394   |
+---------------+----------+



In [27]:
from pyspark.sql.functions import rand

df.select("Weather_Condition").orderBy(rand()).show(15, truncate=False)

+----------------------+
|Weather_Condition     |
+----------------------+
|Fair                  |
|Thunderstorms and Rain|
|Fair                  |
|Cloudy                |
|Partly Cloudy         |
|Fair                  |
|Fair                  |
|Fair                  |
|Clear                 |
|Mostly Cloudy         |
|Partly Cloudy         |
|Fair                  |
|Fair                  |
|Fair                  |
|Scattered Clouds      |
+----------------------+
only showing top 15 rows



In [28]:
from pyspark.sql.functions import col

threshold = 25000 

weather_counts = (
    df.groupBy("Weather_Condition")
      .count()
      .filter(col("count") > threshold)
      .orderBy(col("count").desc())
)

weather_counts.show(100, truncate=False)


+-----------------+-------+
|Weather_Condition|count  |
+-----------------+-------+
|Fair             |2560802|
|Mostly Cloudy    |1016195|
|Cloudy           |817082 |
|Clear            |808743 |
|Partly Cloudy    |698972 |
|Overcast         |382866 |
|Light Rain       |352957 |
|Scattered Clouds |204829 |
|NULL             |173459 |
|Light Snow       |128680 |
|Fog              |99238  |
|Rain             |84331  |
|Haze             |76223  |
|Fair / Windy     |35671  |
|Heavy Rain       |32309  |
+-----------------+-------+



Ympäristömuuttujat omaksi dataframeksi sekä näiden tyhjät arvot ja prosenttiosuudet

In [29]:
environment_cols = [
    "ID", "Amenity", "Give_Way", 
    "Bump", "Crossing", "Junction", "No_Exit", "Railway", "Roundabout",
    "Station", "Stop", "Traffic_Calming", "Traffic_Signal", "Turning_Loop"
]

environment_df = df.select(environment_cols)
environment_df.cache()
environment_df.count()


7728394

In [30]:
from pyspark.sql.functions import col, sum

environment_nulls = environment_df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in environment_df.columns
])

environment_nulls.show()


+---+-------+--------+----+--------+--------+-------+-------+----------+-------+----+---------------+--------------+------------+
| ID|Amenity|Give_Way|Bump|Crossing|Junction|No_Exit|Railway|Roundabout|Station|Stop|Traffic_Calming|Traffic_Signal|Turning_Loop|
+---+-------+--------+----+--------+--------+-------+-------+----------+-------+----+---------------+--------------+------------+
|  0|      0|       0|   0|       0|       0|      0|      0|         0|      0|   0|              0|             0|           0|
+---+-------+--------+----+--------+--------+-------+-------+----------+-------+----+---------------+--------------+------------+



In [31]:
from pyspark.sql.functions import col
import builtins

rows_total = environment_df.count()

data = []
for c in environment_df.columns:
    n = environment_df.filter(col(c).isNull()).count()
    pct = builtins.round(n / rows_total * 100, 2)  
    data.append((c, n, pct))

null_env_df = spark.createDataFrame(data, ["column", "nulls", "pct"])
null_env_df.show(truncate=False)

+---------------+-----+---+
|column         |nulls|pct|
+---------------+-----+---+
|ID             |0    |0.0|
|Amenity        |0    |0.0|
|Give_Way       |0    |0.0|
|Bump           |0    |0.0|
|Crossing       |0    |0.0|
|Junction       |0    |0.0|
|No_Exit        |0    |0.0|
|Railway        |0    |0.0|
|Roundabout     |0    |0.0|
|Station        |0    |0.0|
|Stop           |0    |0.0|
|Traffic_Calming|0    |0.0|
|Traffic_Signal |0    |0.0|
|Turning_Loop   |0    |0.0|
+---------------+-----+---+



Aika-arvoja dataframeksi siten, että näyttää myös viikonpäivän, kuukauden, onko kyseessä viikonloppu sekä eron alkamisajan ja weather_timestampin välillä

In [4]:
from pyspark.sql.functions import (
    to_timestamp, date_format, col, when, round
)

# 1) Muunna ajat timestamp-muotoon
time_df = df.select(
    "ID",
    to_timestamp("Start_Time").alias("Start_Time"),
    to_timestamp("End_Time").alias("End_Time"),
    to_timestamp("Weather_Timestamp").alias("Weather_Timestamp")
)

# 2) Johdetut aikapiirteet
time_df = (
    time_df
    # Kellonaika muodossa HH:mm
    .withColumn("time_hhmm", date_format("Start_Time", "HH:mm"))

    # Viikonpäivä lyhenteenä (Mon, Tue…)
    .withColumn("day_of_week", date_format("Start_Time", "E"))

    # Kuukausi lyhenteenä (Jan, Feb…)
    .withColumn("month", date_format("Start_Time", "MMM"))

    # Viikonloppu: Y/N
    .withColumn(
        "is_weekend",
        when(date_format("Start_Time", "E").isin("Sat", "Sun"), "Y").otherwise("N")
    )

    # Sääviive minuuteissa, pyöristetty 1 desimaaliin
    .withColumn(
        "weather_delay_minutes",
        round((col("Start_Time").cast("long") - col("Weather_Timestamp").cast("long")) / 60, 1)
    )
)

# 3) Näytä 5 ensimmäistä riviä
time_df.show(5, truncate=False)


+---+-------------------+-------------------+-------------------+---------+-----------+-----+----------+---------------------+
|ID |Start_Time         |End_Time           |Weather_Timestamp  |time_hhmm|day_of_week|month|is_weekend|weather_delay_minutes|
+---+-------------------+-------------------+-------------------+---------+-----------+-----+----------+---------------------+
|A-1|2016-02-08 05:46:00|2016-02-08 11:00:00|2016-02-08 05:58:00|05:46    |Mon        |Feb  |N         |-12.0                |
|A-2|2016-02-08 06:07:59|2016-02-08 06:37:59|2016-02-08 05:51:00|06:07    |Mon        |Feb  |N         |17.0                 |
|A-3|2016-02-08 06:49:27|2016-02-08 07:19:27|2016-02-08 06:56:00|06:49    |Mon        |Feb  |N         |-6.6                 |
|A-4|2016-02-08 07:23:34|2016-02-08 07:53:34|2016-02-08 07:38:00|07:23    |Mon        |Feb  |N         |-14.4                |
|A-5|2016-02-08 07:39:07|2016-02-08 08:09:07|2016-02-08 07:53:00|07:39    |Mon        |Feb  |N         |-13.9  

Tarkasteltu eroa pisimmän alku- ja loppuajan välillä. Kaikkein suurimmat erot vuosissa, selittävä tekijä muu kuin tosiasiallinen onnettomuuden kesto. Näitä täytynee miettiä? 

In [33]:
from pyspark.sql.functions import col

time_df = time_df.withColumn(
    "duration_minutes",
    (col("End_Time").cast("long") - col("Start_Time").cast("long")) / 60
)

longest_duration_df = (
    time_df
    .orderBy(col("duration_minutes").desc())
    .select("ID", "Start_Time", "End_Time", "duration_minutes")
    .limit(5)
)

longest_duration_df.show(truncate=False)



+---------+-------------------+-------------------+-----------------+
|ID       |Start_Time         |End_Time           |duration_minutes |
+---------+-------------------+-------------------+-----------------+
|A-5053641|2016-10-21 07:26:00|2022-02-25 17:45:00|2812999.0        |
|A-4810425|2016-10-21 07:26:00|2022-02-25 17:45:00|2812999.0        |
|A-5399002|2018-04-19 09:24:00|2022-07-20 10:49:45|2236405.75       |
|A-4574829|2018-04-19 09:24:00|2022-07-20 09:59:32|2236355.533333333|
|A-4014778|2018-04-19 09:24:00|2022-07-20 09:59:32|2236355.533333333|
+---------+-------------------+-------------------+-----------------+



Kuinka paljon aineistossa yli 8 tunnin, yli 12 tunnin ja yli 24 tunnin eroja alku- ja loppuajan välillä

In [34]:
from pyspark.sql.functions import col

# 8 tuntia = 480 minuuttia
count_over_8h = time_df.filter(col("duration_minutes") > 8 * 60).count()

# 12 tuntia = 720 minuuttia
count_over_12h = time_df.filter(col("duration_minutes") > 12 * 60).count()

# 24 tuntia = 1440 minuuttia
count_over_24h = time_df.filter(col("duration_minutes") > 24 * 60).count()

print("Yli 8 tuntia:", count_over_8h)
print("Yli 12 tuntia:", count_over_12h)
print("Yli 24 tuntia:", count_over_24h)


Yli 8 tuntia: 167991
Yli 12 tuntia: 98069
Yli 24 tuntia: 34976


Miten alkuajat eroavat weather_timestampin ajasta

In [35]:
from pyspark.sql.functions import col, abs

largest_weather_delay_df = (
    time_df
    .orderBy(abs(col("weather_delay_minutes")).desc())
    .select("ID", "Start_Time", "Weather_Timestamp", "weather_delay_minutes")
    .limit(5)
)

largest_weather_delay_df.show(truncate=False)



+---------+-------------------+-------------------+---------------------+
|ID       |Start_Time         |Weather_Timestamp  |weather_delay_minutes|
+---------+-------------------+-------------------+---------------------+
|A-6407392|2021-01-15 00:00:00|2021-01-15 23:53:00|-1433.0              |
|A-6416766|2021-01-15 00:00:00|2021-01-15 23:53:00|-1433.0              |
|A-6525149|2021-01-15 00:01:00|2021-01-15 23:53:00|-1432.0              |
|A-6361171|2021-01-15 00:01:00|2021-01-15 23:53:00|-1432.0              |
|A-2680332|2018-06-30 00:02:43|2018-06-30 23:53:00|-1430.3              |
+---------+-------------------+-------------------+---------------------+



In [20]:
from pyspark.sql.functions import col, floor, abs, concat_ws, lit

time_df = time_df.withColumn(
    "weather_delay_hm",
    concat_ws(
        " ",
        # Tunnit (säilytetään etumerkki)
        (col("weather_delay_minutes") / 60).cast("int"),
        lit("h"),
        # Minuutit (aina positiiviset)
        (abs(col("weather_delay_minutes")) % 60).cast("int"),
        lit("min")
    )
)

time_df.select(
    "ID", 
    "Start_Time",
    "Weather_Timestamp",
    #"weather_delay_minutes",
    "weather_delay_hm"
).show(5, truncate=False)


+---+-------------------+-------------------+----------------+
|ID |Start_Time         |Weather_Timestamp  |weather_delay_hm|
+---+-------------------+-------------------+----------------+
|A-1|2016-02-08 05:46:00|2016-02-08 05:58:00|0 h 12 min      |
|A-2|2016-02-08 06:07:59|2016-02-08 05:51:00|0 h 17 min      |
|A-3|2016-02-08 06:49:27|2016-02-08 06:56:00|0 h 6 min       |
|A-4|2016-02-08 07:23:34|2016-02-08 07:38:00|0 h 14 min      |
|A-5|2016-02-08 07:39:07|2016-02-08 07:53:00|0 h 13 min      |
+---+-------------------+-------------------+----------------+
only showing top 5 rows



In [41]:
from pyspark.sql.functions import col, abs

largest_weather_delay_df = (
    time_df
    .orderBy(abs(col("weather_delay_minutes")).desc())
    .select(
        "ID",
        "Start_Time",
        "Weather_Timestamp",
        #"weather_delay_minutes",
        "weather_delay_hm"
    )
    .limit(5)
)

largest_weather_delay_df.show(truncate=False)




+---------+-------------------+-------------------+----------------+
|ID       |Start_Time         |Weather_Timestamp  |weather_delay_hm|
+---------+-------------------+-------------------+----------------+
|A-6407392|2021-01-15 00:00:00|2021-01-15 23:53:00|-23 h 53 min    |
|A-6416766|2021-01-15 00:00:00|2021-01-15 23:53:00|-23 h 53 min    |
|A-6525149|2021-01-15 00:01:00|2021-01-15 23:53:00|-23 h 52 min    |
|A-6361171|2021-01-15 00:01:00|2021-01-15 23:53:00|-23 h 52 min    |
|A-2680332|2018-06-30 00:02:43|2018-06-30 23:53:00|-23 h 50 min    |
+---------+-------------------+-------------------+----------------+



Lukumäärät yli 3, 8 ja 12 tunnin erolle alkuajan ja weather timestampin välillä

In [42]:
from pyspark.sql.functions import col, abs

# 3 tuntia = 180 minuuttia
count_over_3h = time_df.filter(col("duration_minutes") > 3 * 60).count()

# 8 tuntia = 480 minuuttia
count_over_8h = time_df.filter(col("duration_minutes") > 8 * 60).count()

# 12 tuntia = 720 minuuttia
count_over_12h = time_df.filter(col("duration_minutes") > 12 * 60).count()

print("Yli 3 tuntia:", count_over_3h)
print("Yli 8 tuntia:", count_over_8h)
print("Yli 12 tuntia:", count_over_12h)

Yli 3 tuntia: 1085425
Yli 8 tuntia: 167991
Yli 12 tuntia: 98069


Geograafinen dataframe ja puuttuvat arvot

In [39]:
geo_cols= [
    "ID",
    "Country",
    "State",
    "County",
    "City",
    "Street",
    "Zipcode",
    "Start_Lat",
    "Start_Lng",
    "Distance(mi)",
    "Timezone"
]

geo_df = df.select(geo_cols)
geo_df.cache()
#geo_df.count()
geo_df.show(5, truncate=False)

+---+-------+-----+----------+------------+-------------------------+----------+-----------------+------------------+------------+----------+
|ID |Country|State|County    |City        |Street                   |Zipcode   |Start_Lat        |Start_Lng         |Distance(mi)|Timezone  |
+---+-------+-----+----------+------------+-------------------------+----------+-----------------+------------------+------------+----------+
|A-1|US     |OH   |Montgomery|Dayton      |I-70 E                   |45424     |39.865147        |-84.058723        |0.01        |US/Eastern|
|A-2|US     |OH   |Franklin  |Reynoldsburg|Brice Rd                 |43068-3402|39.92805900000001|-82.831184        |0.01        |US/Eastern|
|A-3|US     |OH   |Clermont  |Williamsburg|State Route 32           |45176     |39.063148        |-84.032608        |0.01        |US/Eastern|
|A-4|US     |OH   |Montgomery|Dayton      |I-75 S                   |45417     |39.747753        |-84.20558199999998|0.01        |US/Eastern|
|A-5|U

In [40]:
from pyspark.sql.functions import col
import builtins

rows_total = geo_df.count()

data = []
for c in geo_df.columns:
    n = geo_df.filter(col(c).isNull()).count()
    pct = builtins.round(n / rows_total * 100, 2)  
    data.append((c, n, pct))

null_geo_df = spark.createDataFrame(data, ["column", "nulls", "pct"])
null_geo_df.show(truncate=False)

+------------+-----+----+
|column      |nulls|pct |
+------------+-----+----+
|ID          |0    |0.0 |
|Country     |0    |0.0 |
|State       |0    |0.0 |
|County      |0    |0.0 |
|City        |253  |0.0 |
|Street      |10869|0.14|
|Zipcode     |1915 |0.02|
|Start_Lat   |0    |0.0 |
|Start_Lng   |0    |0.0 |
|Distance(mi)|0    |0.0 |
|Timezone    |7808 |0.1 |
+------------+-----+----+



In [42]:
from pyspark.sql.functions import col, sum, when

cols_to_check = ["State", "County", "City", "Street"]

missing_exprs = []

for c in cols_to_check:
    missing_count = sum(when(col(c).isNull(), 1).otherwise(0)).alias(f"{c}_missing")
    missing_exprs.append(missing_count)

missing_table = df.select(missing_exprs)
missing_table.show(truncate=False)



+-------------+--------------+------------+--------------+
|State_missing|County_missing|City_missing|Street_missing|
+-------------+--------------+------------+--------------+
|0            |0             |253         |10869         |
+-------------+--------------+------------+--------------+



Valo-olosuhteet sekä columnien tyhjät arvot. Huomioidaan jatkon kannalta merkittävinä columneina Sunrise_Sunset ja Civil_Twilight, muodostetaan näistä kolme ryhmää (Day, Twilight ja Night)

In [6]:
light_cols = [
    "ID",
    "Sunrise_Sunset",
    "Civil_Twilight",
    "Nautical_Twilight",
    "Astronomical_Twilight"
]

light_df = df.select(light_cols)
light_df.cache()

light_df.show(5, truncate=False)




+---+--------------+--------------+-----------------+---------------------+
|ID |Sunrise_Sunset|Civil_Twilight|Nautical_Twilight|Astronomical_Twilight|
+---+--------------+--------------+-----------------+---------------------+
|A-1|Night         |Night         |Night            |Night                |
|A-2|Night         |Night         |Night            |Day                  |
|A-3|Night         |Night         |Day              |Day                  |
|A-4|Night         |Day           |Day              |Day                  |
|A-5|Day           |Day           |Day              |Day                  |
+---+--------------+--------------+-----------------+---------------------+
only showing top 5 rows



In [44]:
from pyspark.sql.functions import col, sum as spark_sum

light_df_nulls = light_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c + "_nulls")
    for c in light_df.columns if c != "ID"
])

light_df_nulls.show(truncate=False)


+--------------------+--------------------+-----------------------+---------------------------+
|Sunrise_Sunset_nulls|Civil_Twilight_nulls|Nautical_Twilight_nulls|Astronomical_Twilight_nulls|
+--------------------+--------------------+-----------------------+---------------------------+
|23246               |23246               |23246                  |23246                      |
+--------------------+--------------------+-----------------------+---------------------------+



In [10]:
from pyspark.sql.functions import when, col

# 1) Luo light_df ja lisää uusi luokittelusarakkeesi
light_df = light_df.withColumn(
    "light_condition",
    when(col("Sunrise_Sunset") == "Day", "Day")
    .when(
        (col("Sunrise_Sunset") == "Night") & (col("Civil_Twilight") == "Day"),
        "Twilight"
    )
    .otherwise("Night")
)

# 2) Luo ryhmitelty DataFrame (oma df, ei pelkkä show)
light_condition_df = (
    light_df
    .groupBy("light_condition")
    .count()
    .withColumnRenamed("count", "accident_count")
)

# 3) Tarkista sisältö
light_condition_df.show()


+---------------+--------------+
|light_condition|accident_count|
+---------------+--------------+
|       Twilight|        361146|
|          Night|       2032695|
|            Day|       5334553|
+---------------+--------------+



Description -columnin sisältö

In [21]:
from pyspark.sql.functions import rand

df.select("ID", "Description").orderBy(rand()).show(15, truncate=False)




+---------+-----------------------------------------------------------------------------------------------------------+
|ID       |Description                                                                                                |
+---------+-----------------------------------------------------------------------------------------------------------+
|A-7394919|At Crowley Rd/Harper Westfall Rd - Accident.                                                               |
|A-6397685|5635 SHELIA --COMMERCE. RP HAS STOP FOLLOWING SV -- ADVS HIS JOB IS ABT 1/2 MILE FROM TC 1020              |
|A-7362023|At CA-237/W Calaveras Blvd - Accident.                                                                     |
|A-7509543|At CA-91/Artesia Fwy - Accident.                                                                           |
|A-704074 |Lane blocked due to crash on I-290 Eisenhower Expy Westbound at Exit 23B Central Ave.                      |
|A-2904432|HOV & #1 lane blocked due to 

Ilmoitetut vakavuusluokat ja niiden määrät

In [5]:
from pyspark.sql.functions import col, sum, when, round, lit

# Kokonaisrivimäärä
total_rows = df.count()

# Lasketaan määrät Sparkissa (Column-objekteina)
severity_counts = df.select(
    sum(when(col("Severity") == 1, 1).otherwise(0)).alias("severity_1"),
    sum(when(col("Severity") == 2, 1).otherwise(0)).alias("severity_2"),
    sum(when(col("Severity") == 3, 1).otherwise(0)).alias("severity_3"),
    sum(when(col("Severity") == 4, 1).otherwise(0)).alias("severity_4"),
    sum(when(col("Severity").isNull(), 1).otherwise(0)).alias("severity_missing")
).withColumn("total_rows", lit(total_rows))

# Muutetaan pystytaulukoksi
severity_long = severity_counts.selectExpr(
    "stack(5, "
    "'Severity 1', severity_1, "
    "'Severity 2', severity_2, "
    "'Severity 3', severity_3, "
    "'Severity 4', severity_4, "
    "'Missing', severity_missing"
    ") as (category, count)",
    "total_rows"
)

# Lisätään prosentit
severity_long = severity_long.withColumn(
    "pct",
    round(col("count") / col("total_rows") * 100, 2)
)

severity_long.show(truncate=False)


+----------+-------+----------+-----+
|category  |count  |total_rows|pct  |
+----------+-------+----------+-----+
|Severity 1|67366  |7728394   |0.87 |
|Severity 2|6156981|7728394   |79.67|
|Severity 3|1299337|7728394   |16.81|
|Severity 4|204710 |7728394   |2.65 |
|Missing   |0      |7728394   |0.0  |
+----------+-------+----------+-----+



Koko aineiston columnit on jaoteltu aihealueittain (muutamaa poikkeusta lukuunottamatta) DataFrameiksi ja tallennetaan Parquet muotoon. 

In [48]:
light_df.write.mode("overwrite").parquet("data/light_df.parquet")
geo_df.write.mode("overwrite").parquet("data/geo_df.parquet")
weather_df.write.mode("overwrite").parquet("data/weather_df.parquet")
time_df.write.mode("overwrite").parquet("data/time_df.parquet")
environment_df.write.mode("overwrite").parquet("data/environment_df.parquet")


Py4JJavaError: An error occurred while calling o1569.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:793)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1249)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1454)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:601)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(F

In [52]:
spark = SparkSession.builder.getOrCreate()
spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS")



'file:///'

In [53]:
spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()


'3.3.4'